# News Pipeline — Week 2 (2.2–2.6)

In an existing Colab session, replace the Python files and rerun sections 3–8. In a fresh session, run sections 1–8.

The implementation is in `news_pipeline.py`. The notebook loads the inputs, runs offline tests, calls the shared `llm.py`, and exports separate evidence. Run sections 1–4 first. Sections 5–8 run the pipeline with the real model and inspect its output. Use the previously tested Colab A100 runtime for nonempty inputs.

Required files: the updated `news_pipeline.py` and `test_news_pipeline.py`, plus the existing integrated `report.py`, shared `llm.py`, and `aapl_news_snapshot_01.json`. The `report.run_news_chain` entry must delegate to `news_pipeline.run_news_chain`, not return the old stub.

For Jason’s cache, also provide `tools.py` and `AAPL_news.json`.

Keep historical JSON files unchanged. This version adds fields and summarizes each article separately, so its outputs and call counts are expected to differ from the old baseline. Passing software checks does not confirm that the factual claims are correct.

## 1. Install dependencies

Keep Colab's installed PyTorch/CUDA. If installation requests a runtime restart,
restart before continuing. No model is loaded in sections 1–4.

In [ ]:
%pip install -q "transformers==4.57.6" "accelerate>=0.26,<2" sentencepiece safetensors pandas yfinance

## 2. Select the working folder

The default matches the Drive folder in your screenshot. Replace the two Python
files there with this update before running. Keep their exact names, without
`(1)` suffixes. If the snapshot is elsewhere, copy it into this folder or change
`INPUT_NAME` in section 4. Set `USE_DRIVE=False` to upload files to a temporary
Colab folder instead. No changes are made to GitHub by this notebook.

In [ ]:
from pathlib import Path
from google.colab import drive, files

USE_DRIVE = True
if USE_DRIVE:
    drive.mount('/content/drive')
    WORK_DIR = Path('/content/drive/MyDrive/AAI520/Final_project/week2')
    if not WORK_DIR.is_dir():
        raise FileNotFoundError('Change WORK_DIR to your actual Drive folder.')
else:
    WORK_DIR = Path('/content/news_week2')
    WORK_DIR.mkdir(exist_ok=True)
    uploaded = files.upload()
    for name, data in uploaded.items():
        (WORK_DIR / Path(name).name).write_bytes(data)

print('Working folder:', WORK_DIR)
print('Files:', sorted(p.name for p in WORK_DIR.iterdir() if p.is_file()))

## 3. Reload source modules and run offline tests

This checks the report entry point as well as the news module. A failure against
`status="stub"` means the folder still contains the old `report.py`; use the
already integrated version. Do not skip failing tests to run the GPU.

In [ ]:
import hashlib
import importlib
import subprocess
import sys

SOURCE_NAMES = ['news_pipeline.py', 'test_news_pipeline.py',
                'report.py', 'llm.py']
for name in SOURCE_NAMES:
    if not (WORK_DIR / name).is_file():
        raise FileNotFoundError(f'Missing {name} in {WORK_DIR}')
sys.path.insert(0, str(WORK_DIR))
for name in ['news_pipeline', 'report', 'test_news_pipeline']:
    sys.modules.pop(name, None)
importlib.invalidate_caches()
import news_pipeline as news
import report

assert Path(news.__file__).resolve() == (WORK_DIR / 'news_pipeline.py').resolve()
assert Path(report.__file__).resolve() == (WORK_DIR / 'report.py').resolve()
assert news.PIPELINE_VERSION == 'news_week2_v15', 'Replace the old news_pipeline.py.'
SOURCE_HASHES = {
    name: hashlib.sha256((WORK_DIR / name).read_bytes()).hexdigest()
    for name in SOURCE_NAMES
}
completed = subprocess.run(
    [sys.executable, str(WORK_DIR / 'test_news_pipeline.py')],
    cwd=WORK_DIR, capture_output=True, text=True,
)
print(completed.stdout)
print(completed.stderr)
completed.check_returncode()
assert report.run_news_chain([])['pipeline_version'] == news.PIPELINE_VERSION
print('Offline checks passed. Pipeline:', news.PIPELINE_VERSION)

## 4. Load and label the input

First use `saved_snapshot` with `RUN_LIMIT=5`. For the later cache test, set
`INPUT_MODE='committed_cache'`, `INPUT_NAME='AAPL_news.json'`, and `RUN_LIMIT=10`.
A missing cache raises an error rather than silently fetching live data.
Choose `live_fetch` explicitly only when you want new data.

`retrieved_at` is kept unchanged. Its `None` cache signal applies to records
returned by `tools.get_news`; an old saved snapshot is labeled separately.
Publication/update timestamps are never converted into event dates.

In [ ]:
import json
from copy import deepcopy

INPUT_MODE = 'saved_snapshot'
INPUT_NAME = 'aapl_news_snapshot_01.json'
TICKER = 'AAPL'
RUN_LIMIT = 5
#INPUT_MODE = "committed_cache"
#INPUT_NAME = "AAPL_news.json"
#TICKER = "AAPL"
#RUN_LIMIT = 10

#BASELINE_NAME = (
#    'results/news_week2_20260928T230405_252322Z/'
#    'news_week2_20260928T230405_252322Z.json'
#)

# Different inputs: skip comparison with the five-article baseline.
BASELINE_NAME = ""


if type(RUN_LIMIT) is not int or RUN_LIMIT < 1:
    raise ValueError('RUN_LIMIT must be a positive integer.')
input_path = WORK_DIR / INPUT_NAME
input_metadata = {'input_mode': INPUT_MODE, 'requested_limit': RUN_LIMIT}
if INPUT_MODE == 'saved_snapshot':
    snapshot = json.loads(input_path.read_text(encoding='utf-8'))
    original_records = snapshot['records']
    if not isinstance(original_records, list):
        raise TypeError('Snapshot records must be a list.')
    if snapshot['records_sha256'] != news.digest(original_records):
        raise ValueError('Snapshot hash mismatch; do not edit the old input.')
    if snapshot['ticker'] != TICKER:
        raise ValueError('Snapshot ticker differs from TICKER.')
    input_metadata['snapshot_records_sha256'] = snapshot['records_sha256']
    input_metadata['snapshot_retrieved_at_utc'] = snapshot.get('retrieved_at_utc')
elif INPUT_MODE in ('committed_cache', 'live_fetch'):
    if not (WORK_DIR / 'tools.py').is_file():
        raise FileNotFoundError('Upload the existing tools.py first.')
    sys.modules.pop('tools', None)
    importlib.invalidate_caches()
    import tools
    if INPUT_MODE == 'committed_cache':
        if input_path.name != f'{TICKER}_news.json' or not input_path.is_file():
            raise FileNotFoundError('Supply the exact ticker_news.json cache file.')
        original_records = tools.get_news(TICKER, data_dir=input_path.parent)
        if any(r.get('retrieved_at') is not None for r in original_records):
            raise ValueError('Cache signal changed; inspect tools.get_news.')
    else:
        live_dir = WORK_DIR / 'live_fetch_input'
        live_dir.mkdir(exist_ok=True)
        if (live_dir / f'{TICKER}_news.json').exists():
            raise ValueError('Choose an empty live_dir to avoid a cache hit.')
        original_records = tools.get_news(TICKER, data_dir=live_dir)
        if any(r.get('retrieved_at') is None for r in original_records):
            raise ValueError('Live fetch lacks retrieved_at; inspect tools.py.')
        input_path = None
    SOURCE_HASHES['tools.py'] = hashlib.sha256(
        (WORK_DIR / 'tools.py').read_bytes()
    ).hexdigest()
else:
    raise ValueError('Choose saved_snapshot, committed_cache, or live_fetch.')

if original_records and len(original_records) < RUN_LIMIT:
    raise ValueError(f'Only {len(original_records)} records; lower RUN_LIMIT explicitly.')
selected_originals = deepcopy(original_records[:RUN_LIMIT])
records = [dict(row, input_mode=INPUT_MODE) for row in selected_originals]
input_metadata.update({
    'available_records': len(original_records), 'selected_records': len(records),
    'selected_original_records_sha256': news.digest(selected_originals),
    'input_file': input_path.name if input_path else None,
    'input_file_sha256': hashlib.sha256(input_path.read_bytes()).hexdigest()
    if input_path else None,
    'selection': 'first RUN_LIMIT records in source order',
})
print(json.dumps(input_metadata, indent=2))
prepared, excluded = news.prepare_news(records)
print('Prepared:', len(prepared), 'Excluded:', excluded)
if not records:
    print('Empty input: the pipeline will return no_data without loading a model.')

## 5. Load the shared model and record the environment

Select Runtime → Change runtime type → A100 GPU for the tested BF16 setup.
The existing helper controls model loading. Record its actual resolved revision;
it is not assumed to be identical to the previous run. Empty or entirely invalid
inputs skip model loading. Restart the runtime if an older helper is still loaded.

In [ ]:
import platform
from importlib.metadata import version, PackageNotFoundError
from time import perf_counter
import torch

for name, expected in SOURCE_HASHES.items():
    assert hashlib.sha256((WORK_DIR / name).read_bytes()).hexdigest() == expected, name
backend = None
model_load_seconds = 0.0
model_info = {'loaded': False}
if prepared:
    if not torch.cuda.is_available() or not torch.cuda.is_bf16_supported():
        raise RuntimeError('Select the tested A100/BF16 Colab runtime first.')
    old_backend = sys.modules.get('llm')
    if old_backend is not None and getattr(old_backend, '_model', None) is not None:
        if getattr(old_backend, '_news_runner_source_sha256', None) != SOURCE_HASHES['llm.py']:
            raise RuntimeError('Restart the runtime to avoid using a stale llm.py.')
    else:
        sys.modules.pop('llm', None)
    import llm as backend
    assert Path(backend.__file__).resolve() == (WORK_DIR / 'llm.py').resolve()
    backend._news_runner_source_sha256 = SOURCE_HASHES['llm.py']
    torch.cuda.synchronize()
    started = perf_counter()
    backend._load()
    backend._model.eval()
    torch.cuda.synchronize()
    model_load_seconds = perf_counter() - started
    model_info = {
        'loaded': True, 'model_id': backend.MODEL_ID,
        'resolved_revision': getattr(backend._model.config, '_commit_hash', None),
        'dtype': str(next(backend._model.parameters()).dtype),
        'device': str(backend._device),
        'context_limit': backend._model.config.max_position_embeddings,
        'attention_implementation': getattr(backend._model.config, '_attn_implementation', None),
        'quantization_config': str(getattr(backend._model.config, 'quantization_config', None)),
    }
packages = {}
for name in ['torch', 'transformers', 'accelerate', 'pandas', 'yfinance']:
    try:
        packages[name] = version(name)
    except PackageNotFoundError:
        packages[name] = None
environment = {
    'python': platform.python_version(), 'platform': 'Google Colab',
    'packages': packages, 'cuda_runtime': torch.version.cuda,
    'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    'gpu_memory_GiB': torch.cuda.get_device_properties(0).total_memory / 1024**3
    if torch.cuda.is_available() else None,
}
print(json.dumps({'environment': environment, 'model': model_info}, indent=2))

## 6. Run the news chain

The direct callback receives `temperature=0`, the system prompt, and each stage's
output limit. A context check runs before each shared-model call; oversized input
is recorded as an error rather than silently truncated. Custom callbacks must
accept the same keyword arguments and enforce their own context limit.

Summaries are generated per article, then combined without another model call.
Five relevant, usable articles normally mean 15 first attempts.
A parse or validation failure permits one corrective retry for that stage.
Backend/context errors are not retried. Actual call counts can therefore vary.
Bare extraction arrays are wrapped in `claims` with an audit entry; all field
checks still run. `raw_response` and `parsed` retain the original output.
Resolved attempts stay in the trace; check `unresolved_errors` separately.

In [ ]:
from datetime import datetime, timezone

for name, expected in SOURCE_HASHES.items():
    assert hashlib.sha256((WORK_DIR / name).read_bytes()).hexdigest() == expected, name
result = None
run_started_at = datetime.now(timezone.utc).isoformat()
if model_info['loaded']:
    torch.cuda.synchronize()
started = perf_counter()
result = report.run_news_chain(
    records, llm=backend.llm if backend is not None else None,
)
if model_info['loaded']:
    torch.cuda.synchronize()
elapsed = perf_counter() - started
result.setdefault('run_metadata', {}).update(input_metadata)
result['run_metadata'].update({
    'started_at_utc': run_started_at,
    'finished_at_utc': datetime.now(timezone.utc).isoformat(),
    'llm_call_mode': 'explicit_shared_callback' if prepared else 'no_model_needed',
    'source_file_sha256': SOURCE_HASHES,
    'model': model_info, 'environment': environment,
    'model_load_seconds': model_load_seconds,
    'pipeline_elapsed_seconds': elapsed,
    'timing_scope': 'pipeline wall time, including validation; excludes explicit model load',
    'offline_tests_returncode': completed.returncode,
})
print('Version:', result['pipeline_version'])
print('Status:', result['status'])
print('Claims:', len(result['claims']), 'Model calls:', len(result['trace']))
print('Pipeline seconds:', round(elapsed, 2))
print(result['summary'])
print('Validation:', json.dumps(result['validation_summary'], indent=2))


## 7. Inspect classification, evidence, coverage, and baseline differences

Review each claim against its supporting text, including negation, company,
numerical scope, units, and dates. Check each summary against the cited claims.
An exact quote or a resolvable ID does not prove semantic support.

The known-issue probes below apply only to the original fixed five-record
snapshot. They are phrase-presence checks, not factual approval or a requirement
to insert details into unrelated news. No model response is manually rewritten.

Check article coverage as well as the phrase probes. `BASELINE_NAME` points
to the previous v3 export by default; adjust it if you moved that file.
Company, metric, date, and sentiment checks are limited software checks,
not a full semantic fact-check. Review the source and numerical scope.


In [ ]:
for item in result['article_results']:
    print(item['article_id'], item['status'], item['classification'])
for claim in result['claims']:
    print(json.dumps({k: claim.get(k) for k in [
        'claim_id', 'statement', 'company', 'metric', 'value', 'unit',
        'event_date', 'period', 'supporting_text', 'url',
    ]}, indent=2, ensure_ascii=False))
for step in result['trace']:
    print(step['stage'], step['article_id'], step['generation_settings'],
          'accepted:', step['schema_valid'], 'attempt:', step.get('attempt'),
          'normalizations:', step.get('normalizations'),
          'recovered:', step.get('resolved_by_retry'), 'error:', step['error'])
print('Coverage:', json.dumps(result['coverage'], indent=2))

review_probes = []
fixed_hash = '796e2f7d3d122a8e08713623569f4cbcc842de13ab9224e37c064965ea936ee2'
if input_metadata.get('snapshot_records_sha256') == fixed_hash:
    checks = [
        ('AAPL_a9c90e3697', 'Music Hall opening', ['music hall', 'open']),
        ('AAPL_a7f18f79cb', 'Watch coverage', ['watch']),
        ('AAPL_a7f18f79cb', 'AirPods coverage', ['airpods']),
    ]
    summary = result['stages']['summarize'] or {'bullets': []}
    for aid, label, phrases in checks:
        article_claims = [c for c in result['claims'] if c['article_id'] == aid]
        ids = {c['claim_id'] for c in article_claims}
        claim_text = ' '.join(c['statement'] for c in article_claims).lower()
        summary_text = ' '.join(b['text'] for b in summary['bullets']
                                if ids.intersection(b['claim_ids'])).lower()
        review_probes.append({
            'issue': label,
            'phrases_in_claims': all(x in claim_text for x in phrases),
            'phrases_in_summary': all(x in summary_text for x in phrases),
            'manual_review': 'pending; phrase presence is not semantic support',
        })
print('Known-issue probes:', json.dumps(review_probes, indent=2))

def counts(run):
    return {
        'status': run.get('status'), 'claims': len(run.get('claims', [])),
        'calls': len(run.get('trace', [])),
        'strict_json_valid': sum(bool(x.get('strict_json_valid')) for x in run.get('trace', [])),
        'schema_valid': sum(bool(x.get('schema_valid')) for x in run.get('trace', [])),
        'errors_all_attempts': sum(bool(x.get('error')) for x in run.get('trace', [])),
        'unresolved_errors': sum(bool(x.get('error')) and not x.get('resolved_by_retry', False)
                                 for x in run.get('trace', [])),
        'normalized_attempts': sum(bool(x.get('normalizations')) for x in run.get('trace', [])),
        'recovered_attempts': sum(bool(x.get('resolved_by_retry')) for x in run.get('trace', [])),
        'articles_with_claims': len({c['article_id'] for c in run.get('claims', [])}),
    }

comparison = {'current': counts(result), 'baseline': None}
baseline_path = WORK_DIR / BASELINE_NAME
if baseline_path.is_file():
    baseline = json.loads(baseline_path.read_text(encoding='utf-8'))
    comparison['baseline'] = counts(baseline)
    comparison['baseline_file_sha256'] = hashlib.sha256(baseline_path.read_bytes()).hexdigest()
else:
    print('Baseline not found. Set BASELINE_NAME to the previous run JSON path to compare.')
print(json.dumps(comparison, indent=2))

## 8. Save and download a separate run

Exports retain the raw model responses, input used, and comparison/probe
results. The manual review worksheet contains an article row for every
processed input, including failures, plus claim and summary rows. Excluded
inputs have separate rows. Human review fields start as pending. Each export has a unique folder
and timestamp; original baseline files remain untouched. Review statuses stay
pending until you inspect the evidence. Clear notebook outputs before committing.
Keep related week-2 changes in one PR under the team's current workflow.

In [ ]:
import csv
import zipfile

assert isinstance(result, dict), 'Run section 6 first.'
for name, expected in SOURCE_HASHES.items():
    assert hashlib.sha256((WORK_DIR / name).read_bytes()).hexdigest() == expected, name
run_id = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
run_dir = WORK_DIR / 'results' / f'news_week2_{run_id}'
run_dir.mkdir(parents=True, exist_ok=False)
result_path = run_dir / f'news_week2_{run_id}.json'
result_path.write_text(json.dumps(result, indent=2, ensure_ascii=False), encoding='utf-8')
(run_dir / 'input_used.json').write_text(json.dumps({
    'ticker': TICKER, 'input_source': INPUT_MODE,
    'records': selected_originals,
    'records_sha256': news.digest(selected_originals),
    'original_input_metadata': input_metadata,
}, indent=2, ensure_ascii=False), encoding='utf-8')
(run_dir / 'comparison_and_review_probes.json').write_text(json.dumps({
    'comparison': comparison, 'phrase_probes': review_probes,
    'manual_review_status': 'pending',
}, indent=2, ensure_ascii=False), encoding='utf-8')
columns = ['kind', 'id', 'text', 'supporting_text', 'source_urls',
           'structured_fields', 'support_review', 'scope_review',
           'coverage_review', 'article_status', 'summary_status',
           'classification', 'stage_errors', 'notes']
with (run_dir / 'manual_review.csv').open('w', encoding='utf-8', newline='') as handle:
    writer = csv.DictWriter(handle, fieldnames=columns)
    writer.writeheader()
    rows_by_id = {row['article_id']: row for row in result['preprocessed']}
    for item in result['article_results']:
        row = rows_by_id[item['article_id']]
        errors = [{k: step.get(k) for k in
                   ['call_id', 'stage', 'attempt', 'error', 'resolved_by_retry']}
                  for step in result['trace']
                  if step['article_id'] == item['article_id'] and step['error']]
        writer.writerow({
            'kind': 'article', 'id': item['article_id'], 'text': row['title'],
            'supporting_text': row['text_raw'], 'source_urls': row['url'],
            'article_status': item['status'],
            'summary_status': item.get('summary_status', 'not_run'),
            'classification': json.dumps(item['classification'], ensure_ascii=False),
            'stage_errors': json.dumps(errors, ensure_ascii=False),
            'support_review': 'pending', 'scope_review': 'pending',
            'coverage_review': 'pending',
        })
    for item in result['excluded_inputs']:
        writer.writerow({
            'kind': 'excluded_input', 'id': str(item['input_index']),
            'text': item['reason'], 'article_status': 'excluded',
            'summary_status': 'not_run', 'coverage_review': 'pending',
            'notes': json.dumps(item, ensure_ascii=False),
        })
    for claim in result['claims']:
        writer.writerow({
            'kind': 'claim', 'id': claim['claim_id'], 'text': claim['statement'],
            'supporting_text': claim['supporting_text'], 'source_urls': claim['url'],
            'structured_fields': json.dumps({k: claim[k] for k in news.STRUCTURED_FIELDS}),
            'support_review': 'pending', 'scope_review': 'pending',
            'coverage_review': 'pending',
        })
    for index, bullet in enumerate((result['stages']['summarize'] or {}).get('bullets', []), 1):
        writer.writerow({
            'kind': 'summary', 'id': f'bullet_{index}', 'text': bullet['text'],
            'supporting_text': json.dumps(bullet['claim_ids']),
            'source_urls': json.dumps(bullet['source_urls']),
            'support_review': 'pending', 'scope_review': 'pending',
            'coverage_review': 'pending',
        })
zip_path = run_dir.with_suffix('.zip')
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(run_dir.iterdir()):
        archive.write(path, path.name)
print('News JSON for mapping:', result_path)
print('Saved:', zip_path)
files.download(str(zip_path))